# Module 4 | Class 2 Assignment
## Binary Classification — Customer Churn Prediction

**Objective:** Logistic Regression modelini qurish, to'liq klassifikatsiya metrikalarini hisoblash va modelni talqin qilish.

---

## Task 1: Ma'lumotlarni Tayyorlash (Data Preparation)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, roc_auc_score,
    classification_report, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

# ── 1. Dataset yuklash ──────────────────────────────────────────────────────
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print("Dataset shakli:", df.shape)
print("\nBirinchi 3 qator:")
df.head(3)

In [ ]:
# ── 2. TotalCharges ustunini tozalash ──────────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
print("TotalCharges NaN qoldi:", df['TotalCharges'].isna().sum())

# ── 3. Maqsad o'zgaruvchisini kodlash ─────────────────────────────────────
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print("\nChurn taqsimoti:")
print(df['Churn'].value_counts())

In [ ]:
# ── 4. Kategorik ustunlarni One-Hot Encoding ───────────────────────────────
cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
df_encoded = pd.get_dummies(df.drop('customerID', axis=1), columns=cat_cols, drop_first=True)
df_encoded = df_encoded.fillna(0)  # qolgan NaN larni to'ldirish
print("Encoded dataset shakli:", df_encoded.shape)
print("Ustunlar soni:", len(df_encoded.columns))

In [ ]:
# ── 5. Train/Test bo'lish (stratifikatsiya bilan) ──────────────────────────
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train churn ulushi: {y_train.mean():.3f}")
print(f"Test  churn ulushi: {y_test.mean():.3f}")

# ── 6. Raqamli ustunlarni standartlashtirish ───────────────────────────────
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_train = X_train.copy()
X_test  = X_test.copy()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])  # faqat train'ga fit
X_test[num_cols]  = scaler.transform(X_test[num_cols])       # test'ga transform

print("\n✅ Ma'lumotlar tayyorlandi!")
print(f"   NaN (train): {X_train.isna().sum().sum()}  |  NaN (test): {X_test.isna().sum().sum()}")

## Task 2: Logistic Regression Modelini O'qitish

In [ ]:
# ── Model yaratish va o'qitish ─────────────────────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# ── Bashorat qilish ────────────────────────────────────────────────────────
y_pred  = model.predict(X_test)            # class labels (0 yoki 1)
y_proba = model.predict_proba(X_test)[:, 1]  # churn ehtimolligi

acc = accuracy_score(y_test, y_pred)
print(f"✅ Model o'qitildi!")
print(f"   Accuracy: {acc:.4f}  ({acc*100:.2f}%)")

## Task 3: To'liq Klassifikatsiya Baholash

### 3a. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test,
    display_labels=['No Churn', 'Churn'],
    colorbar=True, ax=ax, cmap='Blues'
)
ax.set_title('Confusion Matrix — Churn Prediction', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

**Confusion Matrix Talqini:**

Confusion matrix quyidagilarni ko'rsatadi:
- **True Negative (TN):** Model "Churn yo'q" deb to'g'ri bashorat qilgan mijozlar
- **False Positive (FP):** Aslida ketmagan, lekin model "Churn" deb bashorat qilgan mijozlar (**I-tip xato**)
- **False Negative (FN):** Aslida ketgan, lekin model "Churn yo'q" deb bashorat qilgan mijozlar (**II-tip xato** — biznes uchun eng xavfli)
- **True Positive (TP):** Model "Churn" deb to'g'ri bashorat qilgan mijozlar

**Biznes nuqtai nazaridan:** False Negative (FN) eng katta muammo, chunki biz haqiqiy churnerlarni o'tkazib yuboramiz — ularni ushlab qolish imkoniyatini yo'qotamiz. False Positive esa keraksiz marketing xarajatlariga olib keladi.

### 3b. Klassifikatsiya Metrikalari (Classification Report)

In [ ]:
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

### 3c. ROC Egri Chizig'i va AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='royalblue', lw=2,
        label=f'Logistic Regression (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.08, color='royalblue')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Churn Prediction', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"AUC = {auc:.4f}")
print(f"\nAUC = {auc:.3f} — model tasodifiy (~0.5) dan ancha yaxshi,")
print("lekin kamolotga erishish uchun yanada yaxshilash mumkin.")

## Task 4: Modelni Talqin Qilish (Coefficient Analysis)

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
})
coef_df = coef_df.sort_values('Coefficient', ascending=False).reset_index(drop=True)

print("=" * 55)
print("🔴 TOP 5 — Churn ehtimolini OSHIRUVCHI xususiyatlar:")
print("=" * 55)
print(coef_df.head(5).to_string(index=False))

print()
print("=" * 55)
print("🟢 TOP 5 — Mijozni USHLAB TURUVCHI xususiyatlar:")
print("=" * 55)
print(coef_df.tail(5).to_string(index=False))

In [ ]:
# Koeffitsiyentlar vizualizatsiyasi
top10 = pd.concat([coef_df.head(5), coef_df.tail(5)])
colors = ['#e74c3c' if c > 0 else '#27ae60' for c in top10['Coefficient']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10['Feature'], top10['Coefficient'], color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Logistic Regression Koeffitsienti', fontsize=12)
ax.set_title('Churn va Retention Driverlarining Top-10 Koeffitsiyentlari', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

**Koeffitsiyentlar Talqini:**

- **`Contract_Two year` (-1.954)** — Ikki yillik shartnoma churnni eng kuchli pasaytiradi. Bu mantiqiy: uzoq muddatli shartnoma mijozni bog'lab qo'yadi.
- **`Contract_One year` (-1.254)** — Bir yillik shartnoma ham churnni kamaytiradi.
- **`tenure` (-0.508)** — Kompaniyada uzoq yil qolgan mijozlar kamroq ketadi (sodiqlik effekti).
- **`InternetService_Fiber optic` (0.482)** — Fiber optik xizmat foydalanuvchilari ko'proq ketadi — ehtimol narx/sifat muvozanati muammo.
- **`PaymentMethod_Electronic check`** — Elektron to'lov usuli churn bilan bog'liq.

## Task 5: Threshold (Chegara) Tahlili

In [ ]:
results = []
for threshold in [0.3, 0.5, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    results.append({
        'Threshold': threshold,
        'Precision': round(precision_score(y_test, y_pred_t), 4),
        'Recall':    round(recall_score(y_test, y_pred_t), 4),
        'F1':        round(f1_score(y_test, y_pred_t), 4)
    })
    print(f"Threshold: {threshold}")
    print(f"  Precision: {results[-1]['Precision']:.4f}")
    print(f"  Recall:    {results[-1]['Recall']:.4f}")
    print(f"  F1:        {results[-1]['F1']:.4f}")
    print()

thresh_df = pd.DataFrame(results)
print("=" * 42)
print(thresh_df.to_string(index=False))

In [ ]:
# Threshold tahlilini vizualizatsiya qilish
fig, ax = plt.subplots(figsize=(8, 5))
x = [str(r['Threshold']) for r in results]
precision_vals = [r['Precision'] for r in results]
recall_vals    = [r['Recall']    for r in results]
f1_vals        = [r['F1']        for r in results]

ax.plot(x, precision_vals, 'o-', color='#e74c3c', lw=2, ms=8, label='Precision')
ax.plot(x, recall_vals,    's-', color='#27ae60', lw=2, ms=8, label='Recall')
ax.plot(x, f1_vals,        '^-', color='#3498db', lw=2, ms=8, label='F1')
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision / Recall / F1 vs Threshold', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
for i, (p, r, f) in enumerate(zip(precision_vals, recall_vals, f1_vals)):
    ax.annotate(f'{p:.2f}', (x[i], p), textcoords='offset points', xytext=(8, 4), color='#e74c3c', fontsize=9)
    ax.annotate(f'{r:.2f}', (x[i], r), textcoords='offset points', xytext=(8, -12), color='#27ae60', fontsize=9)
plt.tight_layout()
plt.show()

**Threshold Tavsiyasi:**

| Threshold | Precision | Recall | F1 |
|-----------|-----------|--------|-----|
| 0.3 | 0.4822 | 0.7741 | 0.5943 |
| 0.5 | 0.5759 | 0.3662 | 0.4477 |
| 0.7 | 0.6316 | 0.0263 | 0.0505 |

**Biznes maqsadi:** Imkon qadar ko'proq churnerlarni ushlab qolish (Recall ni maksimal qilish)

✅ **Tavsiya: Threshold = 0.3**
- Recall = 0.7741 — ketayotgan mijozlarning katta qismini aniqlaymiz
- Precision pasayadi, ya'ni ba'zi sadiq mijozlarga ham ushlab qolish taklifi yuboriladi — bu nisbatan arzon xarajat
- Threshold = 0.7 da Recall = 0.0263 — juda past, ko'plab churnerlar o'tkazib yuboriladi

---
## Yakuniy Xulosa

| Metrika | Qiymat |
|---------|--------|
| Accuracy | 70.76% |
| AUC | 0.750 |
| Best Threshold (Recall) | 0.3 |

**Model kuchlilari:**
- Uzun muddatli kontrakt va tenure kabi muhim xususiyatlarni to'g'ri aniqladi
- AUC = 0.75 — tasodifiy klassifikatordan (0.5) ancha yaxshi

**Yaxshilash yo'llari:**
- Random Forest yoki Gradient Boosting sinab ko'rish
- Feature engineering (masalan, `charges_per_month = TotalCharges / tenure`)
- SMOTE bilan class imbalance muammosini hal qilish